# Scratch Work for Flow Matching model 

In [1]:
from ase.io import read
from ase import Atoms
from ase.visualize import view

import torch 
from torch import optim
from torch import nn
from torch_geometric.data import Data

from models.gemnet.gemnet_flow import GemNetFlow
from models.gemnet.gemnet_base import GemNetOC
from ase.io import read


/opt/anaconda3/envs/cfm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
contcar = read("POSCAR.vasp")
poscar = read("POSCAR0.vasp")
# Concvert atomic positions to data object
positions = torch.tensor(contcar.get_positions(), dtype=torch.float32)
atomic_numbers = torch.tensor(contcar.get_atomic_numbers(), dtype=torch.long)

## Compile GemNet for flow matching

In [3]:
# GemNet-OC params
model_cfg = {
    "num_targets": 1,
    "num_spherical": 7,
    "num_radial": 128,
    "num_blocks": 4,
    "emb_size_atom": 256,
    "emb_size_edge": 512,
    "emb_size_trip_in": 64,
    "emb_size_trip_out": 64,
    "emb_size_quad_in": 32,
    "emb_size_quad_out": 32,
    "emb_size_aint_in": 64,
    "emb_size_aint_out": 64,
    "emb_size_rbf": 16,
    "emb_size_cbf": 16,
    "emb_size_sbf": 32,
    "num_before_skip": 2,
    "num_after_skip": 2,
    "num_concat": 1,
    "num_atom": 3,
    "num_output_afteratom": 3,
    "num_atom_emb_layers": 0,
    "num_global_out_layers": 2,
    "flow_matching": True,
    "dummy": True,
    "use_pbc": True,
    "scale_backprop_forces": False,
    "cutoff": 12.0,
    "cutoff_qint": 12.0,
    "cutoff_aeaint": 12.0,
    "cutoff_aint": 12.0,
    "max_neighbors": 30,
    "max_neighbors_qint": 8,
    "max_neighbors_aeaint": 20,
    "max_neighbors_aint": 1000,
    "enforce_max_neighbors_strictly": True,
    "rbf": {"name": "gaussian"},
    "rbf_spherical": None,
    "envelope": {"name": "polynomial", "exponent": 5},
    "cbf": {"name": "spherical_harmonics"},
    "sbf": {"name": "legendre_outer"},
    "extensive": True,
    "forces_coupled": True,
    "output_init": "HeOrthogonal",
    "activation": "silu",
    "scale_file": None,
    "quad_interaction": True,
    "atom_edge_interaction": True,
    "edge_atom_interaction": True,
    "atom_interaction": True,
    "scale_basis": True,
}


In [4]:
GemNet = GemNetOC(**model_cfg)

# Count total number of parameters
trainable_params = sum(p.numel() for p in GemNet.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params}")

Trainable parameters: 27550720


In [ ]:
optimizer = optim.Adam(GemNet.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(100): 
  
        # Create batch index (all atoms belong to structure 0)
        x1 = positions 
        batch = torch.zeros(len(x1), dtype=torch.long)

        # Build data objects
        data = Data(pos=x1, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        output = GemNet(data)

        # Compute loss 
        loss = loss_fn(output, torch.tensor(10,  dtype=torch.float32))

        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.01:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    return output

energy = train()



Loss at epoch 0: 81.87963104248047
Loss at epoch 1: 45990.37890625


/opt/anaconda3/envs/cfm/lib/python3.11/site-packages/torch/nn/modules/loss.py:538: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Loss at epoch 2: 112.90562438964844
Loss at epoch 3: 5855.40673828125
Loss at epoch 4: 120.11141967773438
Loss at epoch 5: 38.656288146972656
Loss at epoch 6: 1897.547119140625
Loss at epoch 7: 0.1029474064707756
Loss at epoch 8: 0.6003820300102234
Loss at epoch 9: 1.1405035257339478
Loss at epoch 10: 7.715518951416016
Loss at epoch 11: 2.1114614009857178
Loss at epoch 12: 301.4369201660156
Loss at epoch 13: 11.014863967895508
Loss at epoch 14: 59.3425407409668
Loss at epoch 15: 107.57929229736328
Loss at epoch 16: 695.1014404296875
Loss at epoch 17: 292.6968994140625
Loss at epoch 18: 154.125
Loss at epoch 19: 3.77302885055542
Loss at epoch 20: 13.292862892150879
Loss at epoch 21: 17.903709411621094
Loss at epoch 22: 2.5326757431030273
Loss at epoch 23: 2.140129804611206
Loss at epoch 24: 5.692741870880127
Loss at epoch 25: 0.41004857420921326
Loss at epoch 26: 3.2031610012054443
Loss at epoch 27: 3.862957239151001
Loss at epoch 28: 0.009326320141553879


## Trainer

In [6]:
GemFlow = GemNetFlow(**model_cfg)

In [ ]:
optimizer = optim.Adam(GemFlow.parameters(), 0.001)
loss_fn = nn.MSELoss()

# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for epoch in range(1): 

        x1 = positions 
        x0 = torch.randn_like(x1) #torch.tensor(poscar.get_positions(), dtype=torch.float32)
        t = torch.rand(1)
        xt = (1 - t) * x0 + t * x1
        
        batch = torch.zeros(len(xt), dtype=torch.long)

        # Build data objects
        data = Data(pos=xt, atomic_numbers=atomic_numbers, batch=batch)
        data.cell = torch.tensor(contcar.get_cell().array, dtype=torch.float32).unsqueeze(0) 
        data.pbc = torch.tensor([[True, True, True]])
        data.natoms = torch.tensor([len(positions)], dtype=torch.long)
        data.t = t
        data.tags = torch.zeros(len(positions), dtype=torch.long)

        # Compute u_theta 
        u_theta = GemFlow(data)
        
        # Compute loss 
        loss = loss_fn(u_theta, x1 - x0)
        print(f"Loss at epoch {epoch}: {loss.item()}")

        if loss.item() < 0.01:
            break 

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        


train()

Loss at epoch 0: 5984.5078125


In [ ]:
# Create batch index (all atoms belong to structure 0)
contcar = read("POSCAR.vasp")
positions = torch.tensor(contcar.get_positions(), dtype=torch.float32)
x1 = positions 
x0 = torch.randn_like(x1)
x0_p = torch.tensor([[0, 0, 1], [0, 0, -1]])
print(x1)
print(x0)
print(x0_p)

tensor([[ 0.0000,  0.0000,  0.3686],
        [ 0.0000,  0.0000, -0.3686]])
tensor([[-1.4688,  0.8341,  1.1377],
        [-1.2805,  0.5965,  0.6253]])
tensor([[ 0,  0,  1],
        [ 0,  0, -1]])


## Sampler